# Проект 1



In [2]:
import sys

import pandas as pd
from sqlalchemy import text

try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

подключились к базе как: kausmanova


**Разведочные запросы**

In [3]:
# 1. Справочник цен: когда и как менялась абонентская плата
tariffs = pd.DataFrame(query("""
    select tariff_id, tariff_name, monthly_fee, share_slots, valid_from, valid_to
    from prime.tariffs
    order by tariff_id, valid_from
"""))
tariffs

,tariff_id,tariff_name,monthly_fee,share_slots,valid_from,valid_to
0,1,Базовый,199.00,3,2024-01-01,2026-02-28
1,1,Базовый,249.00,3,2026-03-01,2030-12-31
2,2,Расширенный,399.00,5,2024-01-01,2026-02-28
3,2,Расширенный,429.00,5,2026-02-01,2026-06-30
4,2,Расширенный,449.00,5,2026-03-01,2030-12-31


In [4]:
# 2. Выручка по месяцам и тарифам

# Одна строка = один месяц * один тариф

# Откуда данные:
#   prime.payments p       — платежи (p)
#   prime.subscriptions s  — подписки (s)
#   join ... using (subscription_id) для соединения платежа с его подпиской 
#   У каждого платежа ровно одна подписка, поэтому строк после соединения столько же, сколько платежей
#   subscription_members НЕ беру 

# Что считаю в каждой колонке:
#   month       — месяц платежа: date_trunc обрезает дату до 1-го числа 
#   tariff_id   — тариф: он есть только в подписке, поэтому и нужен join
#   paying_subs — сколько РАЗНЫХ подписок заплатили за месяц (distinct = без повторов)
#   n_success   — сколько списаний прошло
#   n_failed    — сколько списаний не прошло 
#   revenue     — выручка: сумма только успешных списаний

# filter (where ...) — посчитать только строки, где условие выполнено
#   Так в одном запросе получаю и успешные, и неуспешные списания.
#
# group by 1, 2 — сгруппировать по 1-й и 2-й колонке (месяц и тариф)
# order by 1, 2 — отсортировать по месяцу, внутри месяца — по тарифу

revenue = pd.DataFrame(query("""
    select date_trunc('month', p.paid_at)::date                                   as month,
           s.tariff_id,
           count(distinct p.subscription_id) filter (where p.status = 'success')  as paying_subs,
           count(*) filter (where p.status = 'success')                           as n_success,
           count(*) filter (where p.status = 'failed')                            as n_failed,
           sum(p.amount) filter (where p.status = 'success')                      as revenue
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    group by 1, 2
    order by 1, 2
"""))

revenue  # показать таблицу

,month,tariff_id,paying_subs,n_success,n_failed,revenue
0,2025-01-01,1,52282,52338,1822,10415262.00
1,2025-01-01,2,31873,31895,1111,12726105.00
2,2025-02-01,1,60988,61057,2066,12150343.00
3,2025-02-01,2,37114,37143,1249,14820057.00
4,2025-03-01,1,69570,71780,2483,14284220.00
5,2025-03-01,2,42389,43696,1496,17434704.00
6,2025-04-01,1,72756,73195,2571,14565805.00
7,2025-04-01,2,44485,44775,1458,17865225.00
8,2025-05-01,1,77082,79343,2727,15789257.00
9,2025-05-01,2,47159,48638,1723,19406562.00


In [5]:

# 3. Приток и отток подписок по месяцам

# Одна строка результата = один месяц
# Число подписчиков - одни подписки оформляют, другие отменяют, считаю 2 по месяцам

# Запрос состоит из двух частей (with ... as):

# new_subs: сколько подписок оформили за месяц
#   month     - месяц оформления (started_at), обрезанный до 1-го числа
#   new_subs  - всего новых подписок
#   new_trial - из них оформлены в триальный период (is_trial = true)

# cancels: сколько подписок отменили за месяц
#   берем только подписки, у которых есть дата отмены (ended_at is not null),

#   month     - месяц отмены (ended_at)
#   cancelled - всего отмен
#   дальше отмены по причинам:
#     too_expensive  - дорого
#     not_using      - не пользуется
#     payment_failed - не прошла оплата
#     switched       - перешел на другой сервис
#     other          - другое

# В конце соединяю две части по месяцу
# full join нужен, чтобы не потерять месяц, если в нем были только оформления или отмены. Обычный join оставил бы только месяцы, где есть и то и другое
# Если в месяце не было отмен, в колонках отмен будет none

flows = pd.DataFrame(query("""
    with new_subs as (
        select date_trunc('month', started_at)::date as month,
               count(*)                              as new_subs,
               count(*) filter (where is_trial)      as new_trial
        from prime.subscriptions
        group by 1
    ),
    cancels as (
        select date_trunc('month', ended_at)::date                       as month,
               count(*)                                                  as cancelled,
               count(*) filter (where cancel_reason = 'too_expensive')   as too_expensive,
               count(*) filter (where cancel_reason = 'not_using')       as not_using,
               count(*) filter (where cancel_reason = 'payment_failed')  as payment_failed,
               count(*) filter (where cancel_reason = 'switched')        as switched,
               count(*) filter (where cancel_reason = 'other')           as other
        from prime.subscriptions
        where ended_at is not null
        group by 1
    )
    select *
    from new_subs
    full join cancels using (month)
    order by month
"""))

flows  # показать таблицу


,month,new_subs,new_trial,cancelled,too_expensive,not_using,payment_failed,switched,other
0,2024-01-01,2163,353,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-02-01,6148,1047,89.0,25.0,32.0,14.0,13.0,5.0
2,2024-03-01,8791,1512,350.0,91.0,122.0,48.0,59.0,30.0
3,2024-04-01,9704,1638,716.0,171.0,240.0,91.0,144.0,70.0
4,2024-05-01,10670,1691,1122.0,266.0,371.0,179.0,205.0,101.0
5,2024-06-01,10503,1792,1417.0,326.0,505.0,203.0,270.0,113.0
6,2024-07-01,11104,1859,1900.0,459.0,672.0,239.0,339.0,191.0
7,2024-08-01,10976,1841,2294.0,585.0,757.0,296.0,443.0,213.0
8,2024-09-01,10968,1903,2575.0,618.0,838.0,373.0,467.0,279.0
9,2024-10-01,11344,1931,3061.0,701.0,1062.0,431.0,550.0,317.0


In [6]:
# 4. Выручка по каналам продаж по месяцам

# Хотим понять, не тянет ли общую выручку вниз или вверх какой-то один канал

# Одна строка результата запроса = один месяц x один канал

# Откуда беру данные:
#   prime.payments p       - платежи
#   prime.subscriptions s  - подписки, в них есть номер канала partner_id
#   prime.partners pr      - справочник каналов, в нем есть название канала

# Соединения (строк после соединений столько же, сколько платежей, без дв.счет):
#   платеж - подписка по subscription_id (у платежа ровно одна подписка)
#   подписка - канал по partner_id (у подписки ровно один канал)

# Что считаю:
#   month        - месяц платежа до 1-го числа
#   partner_name - название канала
#   paying_subs  - сколько разных подписок заплатили за месяц
#   revenue      - выручка, сумма только успешных списаний

by_partner = pd.DataFrame(query("""
    select date_trunc('month', p.paid_at)::date                                   as month,
           pr.partner_name,
           count(distinct p.subscription_id) filter (where p.status = 'success')  as paying_subs,
           sum(p.amount) filter (where p.status = 'success')                      as revenue
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    join prime.partners pr using (partner_id)
    group by 1, 2
    order by 1, 2
"""))

# Разворачиваем таблицу, чтобы ее было удобно читать (by pivot):
# месяцы по строкам, каналы по столбцам, в ячейках выручка
by_partner.pivot(index="month", columns="partner_name", values="revenue")

partner_name,Автодилер,Банк-партнёр,Маркетплейс,Мобильный оператор,Онлайн-кинотеатр,Прямые продажи,Ритейл-сеть
month,,,,,,,
2025-01-01,3298243.00,3322527.00,3333897.00,3308751.00,3297508.00,3312927.00,3267514.00
2025-02-01,3835639.00,3883689.00,3863158.00,3854785.00,3871599.00,3859570.00,3801960.00
2025-03-01,4544673.00,4548062.00,4564391.00,4511816.00,4549749.00,4513148.00,4487085.00
2025-04-01,4658281.00,4663670.00,4642539.00,4618041.00,4659174.00,4608573.00,4580752.00
2025-05-01,5048248.00,5060235.00,5015778.00,5000410.00,5056752.00,5016133.00,4998263.00
2025-06-01,5101007.00,5124772.00,5069181.00,5059613.00,5133654.00,5080673.00,5064628.00
2025-07-01,5498588.00,5542657.00,5455995.00,5485712.00,5529258.00,5507126.00,5429519.00
2025-08-01,5717567.00,5752467.00,5688545.00,5678000.00,5739885.00,5721558.00,5646094.00
2025-09-01,5750463.00,5739461.00,5672193.00,5700505.00,5772968.00,5733302.00,5669958.00
